In [0]:
from pyspark.sql import functions as F

In [0]:
gold_flags = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Gold/gold_anomaly_flags/"
)

vendor_risk = spark.read.parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/Gold/gold_vendor_risk/"
)


gold_flags.printSchema()
vendor_risk.printSchema()

root
 |-- po_id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- po_date: date (nullable = true)
 |-- invoice_number: string (nullable = true)
 |-- invoice_date: date (nullable = true)
 |-- is_gstin_valid: boolean (nullable = true)
 |-- blacklisted: integer (nullable = true)
 |-- rate_mismatch: integer (nullable = true)
 |-- value_spike: integer (nullable = true)
 |-- name_mismatch: integer (nullable = true)
 |-- duplicate_invoice: integer (nullable = true)
 |-- director_overlap_flag: integer (nullable = true)
 |-- vendor_non_filer: integer (nullable = true)
 |-- missing_ewb: integer (nullable = true)
 |-- state_mismatch: integer (nullable = true)
 |-- late_invoice: integer (nullable = true)
 |-- rule_fail_count: integer (nullable = true)
 |-- rule_score: double (nullable = true)
 |-- is_anomalous: integer (nullable = true)
 |-- severity: string (nullable = true)
 |-- anomaly_code: string (nullable = true)
 |-- gold_created_ts: timestamp (nullable = true)
 |-- se

In [0]:
#Suspicious Subset

suspicious = gold_flags.filter(
    F.col("rule_score") < 70
)

print("Suspicious Records =", suspicious.count())

Suspicious Records = 11355


In [0]:
#Join Vendor Risk 


ai_input_df = (
    gold_flags
    .join(
        vendor_risk.select(
            "vendor_id",
            "risk_tier",
            "anomaly_rate"
        ),
        "vendor_id",
        "left"
    )
)
display(ai_input_df.limit(5))

vendor_id,po_id,po_date,invoice_number,invoice_date,is_gstin_valid,blacklisted,rate_mismatch,value_spike,name_mismatch,duplicate_invoice,director_overlap_flag,vendor_non_filer,missing_ewb,state_mismatch,late_invoice,rule_fail_count,rule_score,is_anomalous,severity,anomaly_code,gold_created_ts,severity_dashboard,rf_prediction,rf_probability,xgb_prediction,xgb_probability,risk_tier,anomaly_rate
V00005,PO0029976,2023-03-23,INV00029976,2023-03-23,true,0,0,1,1,0,1,1,0,1,0,5,50.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,0,0.402,1,0.8921593,Critical,1.0
V00005,PO0020473,2023-04-16,INV00020473,2023-04-16,true,0,0,0,1,0,1,1,0,1,0,4,60.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,1,0.924,1,0.91718984,Critical,1.0
V00005,PO0026128,2023-09-19,INV00026128,2023-09-21,true,0,0,0,1,0,1,1,0,1,0,4,60.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,null,null,null,null,Critical,1.0
V00005,PO0027179,2023-10-18,INV00027179,2023-10-19,true,0,0,1,1,0,1,1,1,1,0,6,40.0,1,High,GST-001,2026-06-17T14:44:20.222Z,Critical,null,null,null,null,Critical,1.0
V00005,PO0042013,2023-10-18,INV00042013,2023-10-19,true,0,0,1,1,0,1,1,0,1,0,5,50.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,null,null,null,null,Critical,1.0


In [0]:
sample_record = ai_input_df.limit(1).collect()[0]

record_dict = sample_record.asDict()

record_dict

{'vendor_id': 'V00005',
 'po_id': 'PO0029976',
 'po_date': datetime.date(2023, 3, 23),
 'invoice_number': 'INV00029976',
 'invoice_date': datetime.date(2023, 3, 23),
 'is_gstin_valid': True,
 'blacklisted': 0,
 'rate_mismatch': 0,
 'value_spike': 1,
 'name_mismatch': 1,
 'duplicate_invoice': 0,
 'director_overlap_flag': 1,
 'vendor_non_filer': 1,
 'missing_ewb': 0,
 'state_mismatch': 1,
 'late_invoice': 0,
 'rule_fail_count': 5,
 'rule_score': 50.0,
 'is_anomalous': 1,
 'severity': 'High',
 'anomaly_code': 'GST-001',
 'gold_created_ts': datetime.datetime(2026, 6, 17, 14, 44, 20, 222507),
 'severity_dashboard': 'High',
 'rf_prediction': 0,
 'rf_probability': 0.402,
 'xgb_prediction': 1,
 'xgb_probability': 0.8921592831611633,
 'risk_tier': 'Critical',
 'anomaly_rate': 1.0}

In [0]:
#AI Analysis Dataset Prepare


ai_df = ai_input_df.select(

    "po_id",
    "vendor_id",
    "rule_score",
    "severity",
    "risk_tier",
    "anomaly_rate",

    "blacklisted",
    "rate_mismatch",
    "value_spike",
    "name_mismatch",
    "duplicate_invoice",
    "vendor_non_filer",
    "missing_ewb",
    "state_mismatch",
    "director_overlap_flag",

    "is_anomalous"

)
display(ai_df.limit(5))

po_id,vendor_id,rule_score,severity,risk_tier,anomaly_rate,blacklisted,rate_mismatch,value_spike,name_mismatch,duplicate_invoice,vendor_non_filer,missing_ewb,state_mismatch,director_overlap_flag,is_anomalous
PO0029976,V00005,50.0,High,Critical,1.0,0,0,1,1,0,1,0,1,1,1
PO0020473,V00005,60.0,High,Critical,1.0,0,0,0,1,0,1,0,1,1,1
PO0026128,V00005,60.0,High,Critical,1.0,0,0,0,1,0,1,0,1,1,1
PO0027179,V00005,40.0,High,Critical,1.0,0,0,1,1,0,1,1,1,1,1
PO0042013,V00005,50.0,High,Critical,1.0,0,0,1,1,0,1,0,1,1,1


In [0]:
#Create Guardrail Function



def validate_record(record):


    if not record.get("po_id"):
        return False

    if not record.get("vendor_id"):
        return False

    if record.get("rule_score") is None:
        return False

    return True



sample = ai_df.limit(1).collect()[0].asDict()

print(validate_record(sample))

True


In [0]:
def build_prompt_v1(record):

    return f"""
You are a GST Fraud Detection Analyst.

Purchase Order Details:

PO ID: {record['po_id']}
Vendor ID: {record['vendor_id']}

Rule Score: {record['rule_score']}
Severity: {record['severity']}

Vendor Risk Tier: {record['risk_tier']}
Vendor Anomaly Rate: {record['anomaly_rate']}

Flags:

Blacklisted: {record['blacklisted']}
Rate Mismatch: {record['rate_mismatch']}
Value Spike: {record['value_spike']}
Name Mismatch: {record['name_mismatch']}
Duplicate Invoice: {record['duplicate_invoice']}
Vendor Non Filer: {record['vendor_non_filer']}
Missing EWB: {record['missing_ewb']}
State Mismatch: {record['state_mismatch']}
Director Overlap: {record['director_overlap_flag']}

Return ONLY JSON:

{{
"risk_level":"",
"anomaly_types_detected":[],
"reasoning":"",
"recommended_action":"",
"confidence_score":0
}}
"""

##Save




In [0]:
suspicious.count()

11355

In [0]:
ai_sample = suspicious.orderBy(
    F.desc("rule_score")
).limit(50)

In [0]:

pdf = ai_sample.toPandas()

pdf.to_csv(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/AI_csv/suspicious_pos.csv",
    index=False
)

In [0]:
# from datetime import datetime

# pdf = suspicious_pos.toPandas()

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# csv_path = f"/Volumes/workspace/default/gst-fraud&anomaly_detection/AI_csv/suspicious_pos_{timestamp}.csv"

# pdf.to_csv(
#     csv_path,
#     index=False
# )

In [0]:
display(pdf.head())

po_id,vendor_id,po_date,invoice_number,invoice_date,is_gstin_valid,blacklisted,rate_mismatch,value_spike,name_mismatch,duplicate_invoice,director_overlap_flag,vendor_non_filer,missing_ewb,state_mismatch,late_invoice,rule_fail_count,rule_score,is_anomalous,severity,anomaly_code,gold_created_ts,severity_dashboard,rf_prediction,rf_probability,xgb_prediction,xgb_probability
PO0048355,V00005,2024-02-15,INV00048355,2024-02-17,true,0,0,0,1,0,1,1,0,1,0,4,60.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,null,null,null,null
PO0023097,V00033,2024-01-06,INV00023097,2024-01-06,true,0,0,0,1,0,1,1,1,0,0,4,60.0,1,High,TAX-001,2026-06-17T14:44:20.222Z,High,null,null,null,null
PO0000729,V00005,2023-06-04,INV00000729,2023-06-07,true,0,0,0,1,0,1,1,0,1,0,4,60.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,null,null,null,null
PO0029449,V00005,2023-12-08,INV00029449,2023-12-08,true,0,0,0,1,0,1,1,0,1,0,4,60.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,null,null,null,null
PO0019961,V00005,2023-02-08,INV00019961,2023-02-10,true,0,0,0,1,0,1,1,0,1,0,4,60.0,1,High,GST-001,2026-06-17T14:44:20.222Z,High,1.0,0.996,1.0,0.92081994
